# 00. Project Fertility PSP v6.2 Final Literature Merged

> **v6.2 제출용 감사 반영본**  
> 이 노트북은 최종 제출 전 실행 가능성, submission 구조 보존, 문헌 해석 안전성을 점검할 수 있도록 구성한 독립 실행형 팀 보고서이다.  
> 입력은 저장소 또는 실행 환경에 배치된 `train.csv`, `test.csv`, `sample_submission.csv` 세 파일로 제한하며, 외부 submission, public score anchor, 이전 제출 후보는 사용하지 않는다.

## 목차

0. 통합본 개요와 독립 실행 원칙
1. 실행 환경 및 Config
2. 데이터 로드와 기본 검증
3. 임상/문헌 기반 변수 그룹 정의
4. Feature Engineering
5. 전처리 파이프라인
6. OOF 검증과 모델 학습
7. Prediction Bank와 앙상블
8. 결과 해석 및 제출용 요약
9. Submission 생성 및 구조 검증
10. 참고문헌 메모

## 0. 통합본 개요와 독립 실행 원칙

- 코드 셀과 실행 흐름은 현재 노트북 안에서 완결되도록 유지한다.
- 문헌 기반 해석은 변수군을 임상적으로 점검하기 위한 보조 설명으로만 사용하며, 특정 feature 수나 고정 성능 수치를 전제하지 않는다.
- 성능 수치, feature 수, 행 개수 등은 실행 결과에 따라 달라질 수 있으므로 고정 문장으로 복사하지 않는다.
- 이 팀 보고서 노트북은 `train.csv`, `test.csv`, `sample_submission.csv`만 사용하는 독립 실행본으로 유지한다.
- 별도 stage 연구, public score anchor, 이전 제출 후보, 외부 submission blend는 사용하지 않는다.


## 1. 실행 환경 및 Config

`Config`는 실행 흐름을 제어하기 위한 단일 설정 객체이다. 기본 입력 디렉터리는 `data/`이며, `PSP_DATA_DIR`, `PSP_OUTPUT_DIR`, `PSP_RUN_MODE` 환경변수로 실행 검증 스크립트에서만 경로와 smoke/full 모드를 바꿀 수 있다. 산출물 디렉터리는 기본적으로 `outputs/team_report_v6_2/`이고, 제출 파일명은 `submission_team_report_v6_2_...csv` 형식으로 생성한다.


In [ ]:
from __future__ import annotations

import json
import os
import random
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")


@dataclass(frozen=True)
class Config:
    seed: int = 42
    run_mode: str = os.environ.get("PSP_RUN_MODE", "full")  # full: 제출 후보 생성 / smoke: 실행 검증용 축소 실행
    n_splits: int = int(os.environ.get("PSP_N_SPLITS", "5"))
    data_dir: Path = Path(os.environ.get("PSP_DATA_DIR", "data"))
    train_path: Path = Path("train.csv")
    test_path: Path = Path("test.csv")
    sample_submission_path: Path = Path("sample_submission.csv")
    output_dir: Path = Path(os.environ.get("PSP_OUTPUT_DIR", "outputs/team_report_v6_2"))
    target_candidates: tuple[str, ...] = (
        "임신 성공 여부",
        "pregnancy_success",
        "target",
        "label",
        "y",
    )
    id_candidates: tuple[str, ...] = (
        "ID",
        "id",
        "시술 ID",
        "환자 ID",
    )
    submission_prefix: str = "submission_team_report_v6_2"


CFG = Config()
CFG.output_dir.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)

print(f"Run mode: {CFG.run_mode}")
print(f"Data directory: {CFG.data_dir}")
print(f"Output directory: {CFG.output_dir}")


## 2. 데이터 로드와 기본 검증

본 노트북은 외부 중간 산출물이나 이전 제출 파일을 사용하지 않는다. 입력은 `train.csv`, `test.csv`, `sample_submission.csv` 세 파일로 제한한다. 기본적으로 `data/` 아래의 세 파일을 찾으며, 실행 검증 스크립트는 동일한 파일명을 가진 별도 smoke fixture 디렉터리를 `PSP_DATA_DIR`로 지정할 수 있다.


In [ ]:
def resolve_path(base_dir: Path, path: Path) -> Path:
    candidate = path if path.is_absolute() else base_dir / path
    if not candidate.exists():
        raise FileNotFoundError(f"Required input file not found: {candidate}")
    return candidate


def find_first_existing(columns: pd.Index, candidates: tuple[str, ...]) -> str | None:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


train_file = resolve_path(CFG.data_dir, CFG.train_path)
test_file = resolve_path(CFG.data_dir, CFG.test_path)
sample_submission_file = resolve_path(CFG.data_dir, CFG.sample_submission_path)

train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
sample_submission = pd.read_csv(sample_submission_file)

if CFG.run_mode == "smoke":
    train = train.head(min(len(train), 40)).copy()
    test = test.head(min(len(test), 12)).copy()
    sample_submission = sample_submission.head(len(test)).copy()

target_col = find_first_existing(train.columns, CFG.target_candidates)
if target_col is None:
    raise ValueError(f"Target column not found. Candidates: {CFG.target_candidates}")

id_col = find_first_existing(test.columns, CFG.id_candidates)
submission_id_col = find_first_existing(sample_submission.columns, CFG.id_candidates)

print("train path:", train_file)
print("test path:", test_file)
print("sample_submission path:", sample_submission_file)
print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)
print("target column:", target_col)
print("test id column:", id_col)
print("sample_submission id column:", submission_id_col)
print("sample_submission columns:", sample_submission.columns.tolist())

assert train_file.name == "train.csv"
assert test_file.name == "test.csv"
assert sample_submission_file.name == "sample_submission.csv"
assert target_col in train.columns, "target column must exist in train"
assert len(test) == len(sample_submission), "test and sample_submission row counts must match"
assert len(sample_submission.columns) >= 1, "sample_submission must have at least one column"
if submission_id_col is not None and id_col is not None:
    assert sample_submission[submission_id_col].astype(str).tolist() == test[id_col].astype(str).tolist(), (
        "sample_submission ID order must match test ID order"
    )

print("Input sanity checks passed.")


## 3. 임상/문헌 기반 변수 그룹 정의

변수 그룹은 모델 입력을 임상적으로 점검하기 위한 보조 구조이다. 여성 연령, 난임 원인, 시술 유형, 배아/난자/이식 관련 변수는 IVF/ICSI 및 FET outcome 해석에서 반복적으로 논의되는 축이므로 결과 해석 시 과도한 인과 해석을 피하고 예측 변수로서의 의미를 중심으로 기술한다 (Murugappan et al., 2023; Carson & Kallen, 2021; Liu et al., 2021).


In [ ]:
CLINICAL_KEYWORDS = {
    "age": ["나이", "연령", "age"],
    "infertility": ["난임", "불임", "infertility", "원인"],
    "procedure": ["시술", "IVF", "ICSI", "IUI", "FET", "FER", "해동", "동결"],
    "embryo": ["배아", "embryo", "blastocyst", "포배"],
    "oocyte": ["난자", "oocyte", "egg"],
    "sperm": ["정자", "sperm", "male"],
    "transfer": ["이식", "transfer"],
}


def match_columns_by_keywords(columns: pd.Index, keywords: list[str]) -> list[str]:
    lowered = {col: str(col).lower() for col in columns}
    return [
        col for col, low in lowered.items()
        if any(keyword.lower() in low for keyword in keywords)
    ]


feature_columns = [col for col in train.columns if col != target_col]
if id_col in feature_columns:
    feature_columns.remove(id_col)

clinical_groups = {
    group: match_columns_by_keywords(pd.Index(feature_columns), keywords)
    for group, keywords in CLINICAL_KEYWORDS.items()
}

for group, cols in clinical_groups.items():
    print(f"{group:>12}: {len(cols)} columns")
    print(cols[:10])


## 4. Feature Engineering

v6 계열의 실행 구조에 맞춰 결측 패턴을 명시적 feature로 분리한다. v5.1에서 사용된 `process_missing_count`류의 일반 결측 해석은 v6 변수명에 맞춰 `f_embryo_missing_count`, `f_embryo_all_missing` 등으로 정리한다. 배아 관련 변수의 결측 여부는 단순 누락이라기보다 시술 유형·배아 배양/이식 단계·기록 방식 차이를 함께 반영할 수 있으므로, 결측 그 자체를 임상적 실패 원인으로 단정하지 않는다.


In [ ]:
def add_missing_pattern_features(df: pd.DataFrame, reference_columns: list[str]) -> pd.DataFrame:
    out = df.copy()

    embryo_cols = match_columns_by_keywords(pd.Index(reference_columns), CLINICAL_KEYWORDS["embryo"])
    oocyte_cols = match_columns_by_keywords(pd.Index(reference_columns), CLINICAL_KEYWORDS["oocyte"])
    transfer_cols = match_columns_by_keywords(pd.Index(reference_columns), CLINICAL_KEYWORDS["transfer"])
    procedure_cols = match_columns_by_keywords(pd.Index(reference_columns), CLINICAL_KEYWORDS["procedure"])

    grouped_missing_specs = {
        "f_embryo_missing_count": embryo_cols,
        "f_oocyte_missing_count": oocyte_cols,
        "f_transfer_missing_count": transfer_cols,
        "f_procedure_missing_count": procedure_cols,
    }

    for feature_name, cols in grouped_missing_specs.items():
        available = [col for col in cols if col in out.columns]
        if available:
            out[feature_name] = out[available].isna().sum(axis=1)
        else:
            out[feature_name] = 0

    available_embryo_cols = [col for col in embryo_cols if col in out.columns]
    out["f_embryo_all_missing"] = (
        out[available_embryo_cols].isna().all(axis=1).astype(int)
        if available_embryo_cols else 0
    )
    available_reference_columns = [col for col in reference_columns if col in out.columns]
    out["f_any_clinical_missing_count"] = out[available_reference_columns].isna().sum(axis=1)
    out["f_any_clinical_missing_ratio"] = out["f_any_clinical_missing_count"] / max(len(available_reference_columns), 1)

    return out


test_ids = test[id_col].copy() if id_col is not None and id_col in test.columns else None
train_fe = add_missing_pattern_features(train.drop(columns=[target_col]), feature_columns)
test_fe = add_missing_pattern_features(test, feature_columns)

if id_col is not None and id_col in train_fe.columns:
    train_fe = train_fe.drop(columns=[id_col])
if id_col is not None and id_col in test_fe.columns:
    test_fe = test_fe.drop(columns=[id_col])

X = train_fe.copy()
y = train[target_col].copy()
X_test = test_fe.reindex(columns=X.columns, fill_value=np.nan)

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("Engineered missing features:", [col for col in X.columns if col.startswith("f_")])


## 5. 전처리 파이프라인

범주형 변수는 빈도/순서의 임의성을 최소화하기 위해 `OrdinalEncoder`와 결측 대체를 결합한다. 수치형 변수는 중앙값 대체를 사용한다. 이 단계는 모델별 학습 fold 내부에서만 적합되어 OOF 검증 누수를 방지한다.


In [ ]:
def build_preprocessor(X_frame: pd.DataFrame) -> ColumnTransformer:
    categorical_cols = X_frame.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [col for col in X_frame.columns if col not in categorical_cols]

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


preprocessor = build_preprocessor(X)
print(preprocessor)


## 6. OOF 검증과 모델 학습

OOF 검증은 제출 파일 생성 전 내부 일관성을 확인하기 위한 절차이다. IVF/ART 예측 모델은 데이터셋, 환자군, 시술 프로토콜에 따라 성능이 달라질 수 있으므로, 단일 점수보다 fold 간 변동성과 임상적으로 설명 가능한 변수 패턴을 함께 확인한다 (Liu et al., 2021; Fu et al., 2025).


In [ ]:
def make_models(seed: int) -> dict[str, Pipeline]:
    base_preprocessor = build_preprocessor(X)
    if CFG.run_mode == "smoke":
        hgb_max_iter = 10
        tree_estimators = 10
    else:
        hgb_max_iter = 300
        tree_estimators = 500

    return {
        "hist_gradient_boosting": Pipeline(steps=[
            ("preprocess", clone(base_preprocessor)),
            ("model", HistGradientBoostingClassifier(
                learning_rate=0.04,
                max_iter=hgb_max_iter,
                l2_regularization=0.05,
                random_state=seed,
            )),
        ]),
        "extra_trees": Pipeline(steps=[
            ("preprocess", clone(base_preprocessor)),
            ("model", ExtraTreesClassifier(
                n_estimators=tree_estimators,
                min_samples_leaf=3,
                max_features="sqrt",
                n_jobs=-1,
                random_state=seed,
            )),
        ]),
        "random_forest": Pipeline(steps=[
            ("preprocess", clone(base_preprocessor)),
            ("model", RandomForestClassifier(
                n_estimators=tree_estimators,
                min_samples_leaf=4,
                max_features="sqrt",
                n_jobs=-1,
                random_state=seed,
            )),
        ]),
    }


def predict_positive_probability(model: Pipeline, X_frame: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_frame)[:, 1]
    decision = model.decision_function(X_frame)
    return 1 / (1 + np.exp(-decision))


models = make_models(CFG.seed)
min_class_count = int(y.value_counts().min())
effective_splits = min(CFG.n_splits, min_class_count)
if CFG.run_mode == "smoke":
    effective_splits = min(effective_splits, 2)
if effective_splits < 2:
    raise ValueError("At least two samples per target class are required for OOF validation.")

skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=CFG.seed)
print(f"Effective CV splits: {effective_splits}")

oof_bank = pd.DataFrame(index=X.index)
prediction_bank = pd.DataFrame(index=X_test.index)
cv_records = []

for model_name, model in models.items():
    oof_pred = np.zeros(len(X), dtype=float)
    test_pred_folds = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        fold_model = clone(model)
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        fold_model.fit(X_tr, y_tr)
        va_pred = predict_positive_probability(fold_model, X_va)
        test_pred = predict_positive_probability(fold_model, X_test)

        oof_pred[va_idx] = va_pred
        test_pred_folds.append(test_pred)

        fold_auc = roc_auc_score(y_va, va_pred)
        cv_records.append({"model": model_name, "fold": fold, "auc": fold_auc})
        print(f"{model_name} fold {fold}: AUC={fold_auc:.5f}")

    oof_bank[model_name] = oof_pred
    prediction_bank[model_name] = np.mean(test_pred_folds, axis=0)
    model_auc = roc_auc_score(y, oof_pred)
    print(f"{model_name} OOF AUC: {model_auc:.5f}")

cv_summary = pd.DataFrame(cv_records)
cv_summary_path = CFG.output_dir / "cv_summary_v6_2.csv"
cv_summary.to_csv(cv_summary_path, index=False)
print(f"Saved CV summary: {cv_summary_path}")


## 7. Prediction Bank와 앙상블

`prediction_bank`는 v6 실행 흐름의 핵심 산출물이다. 각 모델의 test 예측을 하나의 테이블로 보관한 뒤, OOF AUC를 기반으로 가중 평균을 계산한다. 성능 수치는 실행 환경과 데이터 버전에 따라 달라질 수 있으므로, 보고서에는 현재 실행 결과를 확인한 뒤 기재한다.


In [ ]:
def auc_weighted_ensemble(oof_frame: pd.DataFrame, pred_frame: pd.DataFrame, y_true: pd.Series) -> tuple[np.ndarray, np.ndarray, pd.Series]:
    aucs = oof_frame.apply(lambda col: roc_auc_score(y_true, col), axis=0)
    centered = (aucs - 0.5).clip(lower=0)
    if centered.sum() == 0:
        weights = pd.Series(1 / len(aucs), index=aucs.index)
    else:
        weights = centered / centered.sum()

    oof_ensemble = np.average(oof_frame.values, axis=1, weights=weights.values)
    test_ensemble = np.average(pred_frame.values, axis=1, weights=weights.values)
    return oof_ensemble, test_ensemble, weights


oof_ensemble, test_ensemble, ensemble_weights = auc_weighted_ensemble(oof_bank, prediction_bank, y)
ensemble_auc = roc_auc_score(y, oof_ensemble)

prediction_bank["ensemble_v6_2"] = test_ensemble
oof_bank["ensemble_v6_2"] = oof_ensemble

prediction_bank_path = CFG.output_dir / "prediction_bank_v6_2.csv"
oof_bank_path = CFG.output_dir / "oof_bank_v6_2.csv"
weights_path = CFG.output_dir / "ensemble_weights_v6_2.csv"

prediction_bank.to_csv(prediction_bank_path, index=False)
oof_bank.to_csv(oof_bank_path, index=False)
ensemble_weights.rename("weight").to_csv(weights_path)

print("OOF ensemble AUC:", round(ensemble_auc, 6))
print("Ensemble weights:")
print(ensemble_weights)
print(f"Saved prediction bank: {prediction_bank_path}")
print(f"Saved OOF bank: {oof_bank_path}")
print(f"Saved weights: {weights_path}")


## 8. 결과 해석 및 제출용 요약

### 8.1 모델 관점 해석

- 본 노트북의 평가는 OOF 기반 내부 검증 결과를 우선 확인한다. fold 간 편차가 크다면 단일 평균 점수보다 데이터 분할 민감성이 더 중요한 신호일 수 있다.
- 앙상블은 서로 다른 tree 계열 모델의 예측을 `prediction_bank`에 저장한 뒤 OOF 성능 기반 가중 평균으로 결합한다. 이는 특정 모델 하나의 우연한 fold 적합에 제출 결과가 과도하게 의존하는 것을 완화하기 위한 절차이다.
- 성능 수치와 feature 개수는 실행 시점의 데이터/환경에 따라 달라질 수 있으므로, 제출 보고서에는 위 코드 셀의 출력값을 확인한 뒤 반영한다.

### 8.2 임상 변수 해석

- 여성 연령 및 난소 기능과 관련된 변수군은 IVF/ICSI outcome을 설명할 때 자주 언급되는 임상 축이다. 다만 본 모델에서는 원인 추정이 아니라 예측 신호로 해석해야 한다 (Murugappan et al., 2023; Carson & Kallen, 2021).
- 배아 발달 단계, 이식 관련 변수, 동결/해동 배아 이식 여부는 착상 가능성과 치료 프로토콜 차이를 반영할 수 있다. 따라서 `f_embryo_missing_count`, `f_embryo_all_missing` 같은 결측 패턴 feature는 배아 정보의 부재 자체보다 시술 유형과 기록 구조 차이를 함께 나타내는 보조 신호로 해석한다 (Liu et al., 2021; Zhao et al., 2022).
- ICSI 및 남성 요인 관련 변수군은 수정 방식과 난임 원인 구성을 반영할 수 있으나, 대회 데이터만으로 개별 환자의 생물학적 원인을 단정하지 않는다 (Chen et al., 2022; Murugappan et al., 2023).

### 8.3 제출용 한계 문장

본 분석은 제공된 tabular 데이터만으로 임신 성공 여부를 예측하는 모델링 결과이며, 치료 의사결정을 직접 대체하지 않는다. IVF/ART 결과는 환자 연령, 난소 예비력, 배아 상태, 자궁내막 환경, 시술 프로토콜 등 복합 요인의 영향을 받을 수 있으므로, 모델 예측은 임상적 맥락과 함께 보조적으로 해석해야 한다 (Carson & Kallen, 2021; Mohamed Salih et al., 2026).


## 9. Submission 생성 및 구조 검증

제출 파일은 `sample_submission.csv`의 컬럼 구조와 ID 순서를 보존하고, 예측 대상 컬럼에 `ensemble_v6_2` 값을 채운다. 파일명은 `submission_team_report_v6_2_YYYYMMDD_HHMMSS.csv` 형식으로 저장한다. 저장 직전과 직후에는 row 수, ID 순서, null 여부, 확률 범위를 확인한다.


In [ ]:
def infer_submission_target_column(sample: pd.DataFrame) -> str:
    non_id_columns = [col for col in sample.columns if col not in CFG.id_candidates]
    if len(non_id_columns) == 1:
        return non_id_columns[0]
    for candidate in CFG.target_candidates:
        if candidate in sample.columns:
            return candidate
    return sample.columns[-1]


def validate_submission_frame(submission_frame: pd.DataFrame, sample_frame: pd.DataFrame, probability_col: str) -> None:
    assert len(submission_frame) == len(sample_frame), "submission row count must match sample_submission"
    assert submission_frame.columns.tolist() == sample_frame.columns.tolist(), "submission columns must match sample_submission"

    sample_id_col = find_first_existing(sample_frame.columns, CFG.id_candidates)
    if sample_id_col is not None:
        assert submission_frame[sample_id_col].astype(str).tolist() == sample_frame[sample_id_col].astype(str).tolist(), (
            "submission ID column order must match sample_submission"
        )

    assert probability_col in submission_frame.columns, "probability column must exist in submission"
    assert not submission_frame[probability_col].isna().any(), "probability column must not contain null values"
    assert submission_frame[probability_col].between(0, 1).all(), "probability values must be within [0, 1]"


submission = sample_submission.copy()
submission_target_col = infer_submission_target_column(submission)
submission[submission_target_col] = np.clip(test_ensemble, 0, 1)

if id_col is not None and submission_id_col is not None and test_ids is not None:
    submission[submission_id_col] = test_ids.values

validate_submission_frame(submission, sample_submission, submission_target_col)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
submission_path = CFG.output_dir / f"{CFG.submission_prefix}_{timestamp}.csv"
submission.to_csv(submission_path, index=False)

saved_submission = pd.read_csv(submission_path)
validate_submission_frame(saved_submission, sample_submission, submission_target_col)

print("submission target column:", submission_target_col)
print("submission row count:", len(submission))
print("sample_submission row count:", len(sample_submission))
print("ID order preserved:", submission_id_col is None or submission[submission_id_col].astype(str).tolist() == sample_submission[submission_id_col].astype(str).tolist())
print("probability null count:", int(submission[submission_target_col].isna().sum()))
print("probability range:", float(submission[submission_target_col].min()), float(submission[submission_target_col].max()))
print("submission path:", submission_path)
submission.head()


## 10. 참고문헌 메모

노트북 본문에는 필요한 위치에만 간결한 `(저자 또는 기관, 연도)` 형식으로 표기하였다.

- Carson, S. A., & Kallen, A. N. (2021). Diagnosis and Management of Infertility: A Review.
- Chen, H. H., et al. (2022). Clinical predictors of embryo quality among women of advanced age receiving ICSI cycles.
- Fu, X., et al. (2025). Enhancing frozen-thawed embryo transfer outcomes and treatment personalization through machine learning models.
- Liu, Y., et al. (2021). Multifactor prediction of embryo transfer outcomes based on a machine learning algorithm.
- Mohamed Salih, E. L., et al. (2026). Clinical predictors of successful pregnancy after IVF: A comprehensive systematic review of evidence.
- Murugappan, G., et al. (2023). IVF and ICSI predictive factors: female age, ovarian reserve, male age, and male factor.
- Zhao, J., et al. (2022). Nomogram prediction for clinical pregnancy in freeze-thawed embryo transfer.
